In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import joblib
import re

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.bleu_score import SmoothingFunction

In [ ]:
clean_df = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/Resume-Matching/clean_dataset.csv"
)

resume_embeddings = np.load(
    "/content/drive/MyDrive/Colab Notebooks/Resume-Matching/resume_embeddings.npy"
)

job_embeddings = np.load(
    "/content/drive/MyDrive/Colab Notebooks/Resume-Matching/job_embeddings.npy"
)

xgb_model = joblib.load(
    "/content/drive/MyDrive/Colab Notebooks/Resume-Matching/resume_matching_xgboost.pkl"
)

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("✅ Everything Loaded Successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Everything Loaded Successfully!


In [ ]:
print(clean_df.shape)

print(resume_embeddings.shape)

print(job_embeddings.shape)

print(type(xgb_model))

(9544, 18)
(9544, 384)
(9544, 384)
<class 'xgboost.sklearn.XGBRegressor'>


In [ ]:
import re

def extract_experience(text):

    text = text.lower()

    patterns = [

        r'(\d+)\+?\s+years?',
        r'(\d+)\+?\s+yrs?',
        r'(\d+)\+?\s+year',

    ]

    for pattern in patterns:

        match = re.search(pattern, text)

        if match:
            years = int(match.group(1))

            if years <= 50:        # Ignore phone numbers
                return years

    return 0

In [ ]:
print(extract_experience("Experience : 3 years"))

print(extract_experience("Minimum 2+ years required"))

print(extract_experience("Worked for 5 yrs"))

print(extract_experience("Phone : 9032225954"))

3
2
5
0


In [ ]:
def calculate_semantic_similarity(resume, job):

    resume_embedding = embedding_model.encode([resume])

    job_embedding = embedding_model.encode([job])

    similarity = cosine_similarity(
        resume_embedding,
        job_embedding
    )[0][0]

    return similarity

In [ ]:
resume = "Python Machine Learning SQL"

job = "Python SQL Machine Learning"

print(calculate_semantic_similarity(resume, job))

0.9645742


In [ ]:
def calculate_bleu(resume, job):

    smoothie = SmoothingFunction().method1

    score = sentence_bleu(
        [job.lower().split()],
        resume.lower().split(),
        smoothing_function=smoothie
    )

    return score

In [ ]:
resume = "Python Machine Learning SQL"

job = "Python SQL Machine Learning"

print(calculate_bleu(resume, job))

0.20205155046766235


In [ ]:
def calculate_career_similarity(career_objective, job_title):

    career_embedding = embedding_model.encode([career_objective])

    job_embedding = embedding_model.encode([job_title])

    similarity = cosine_similarity(
        career_embedding,
        job_embedding
    )[0][0]

    return similarity

In [ ]:
career = "Looking for an AI Engineer role"

job = "Machine Learning Engineer"

print(calculate_career_similarity(career, job))

0.60492885


In [ ]:
def predict_match(resume_text,
                  career_objective,
                  job_description,
                  experience_text):

    semantic = calculate_semantic_similarity(
        resume_text,
        job_description
    )

    career = calculate_career_similarity(
        career_objective,
        job_description
    )

    bleu = calculate_bleu(
        resume_text,
        job_description
    )

    experience = extract_experience(
        experience_text
    )

    sample = pd.DataFrame({
        "semantic_similarity": [semantic],
        "career_similarity": [career],
        "required_experience": [experience],
        "bleu_score": [bleu]
    })

    prediction = xgb_model.predict(sample)[0]

    print("=" * 60)
    print("              AI RESUME MATCH REPORT")
    print("=" * 60)

    print(f"\n🎯 Match Score : {prediction*100:.2f}%")

    print("\n-------------------------------")

    print(f"🧠 Semantic Similarity : {semantic:.3f}")
    print(f"🎯 Career Similarity   : {career:.3f}")
    print(f"📝 BLEU Score          : {bleu:.3f}")
    print(f"💼 Experience          : {experience} years")

    print("\n===============================")

    if prediction >= 0.80:
        print("✅ Recommendation : Strong Match")
    elif prediction >= 0.60:
        print("🟡 Recommendation : Good Match")
    else:
        print("❌ Recommendation : Weak Match")

    resume_skills = extract_skills(resume_text)
    job_skills = extract_skills(job_description)
    missing = job_skills - resume_skills
    print("\n📌 Missing Skills")

    if len(missing) == 0:
        print("None 🎉")
    else:
        for skill in sorted(missing):
            print("-", skill)

    return float(prediction)


In [ ]:
!pip install pypdf

In [ ]:
from pypdf import PdfReader

In [109]:
def extract_text_from_pdf(pdf_path):

    reader = PdfReader(pdf_path)

    text = ""

    for page in reader.pages:
        text += page.extract_text() + "\n"

    return text

In [90]:
import re

def extract_name(text):

    lines = text.split("\n")

    banned = [
        "PROJECT",
        "PROJECTS",
        "EXPERIENCE",
        "SUMMARY",
        "EDUCATION",
        "SKILLS",
        "CERTIFICATIONS",
        "RESPONSIBILITIES",
        "PROFILE",
        "CONTACT"
    ]

    for line in lines:

        line = line.strip()

        if len(line) == 0:
            continue

        if "@" in line:
            continue

        if re.search(r"\d", line):
            continue

        words = line.split()

        if len(words) < 2 or len(words) > 4:
            continue

        if any(word in banned for word in words):
            continue

        if line.upper() == line:
            return line.title()

    return "Not Found"

In [102]:
def extract_email(text):

    pattern = r'[\w\.-]+@[\w\.-]+\.\w+'

    match = re.search(pattern, text)

    if match:
        return match.group()

    return "Not Found"

In [101]:
def extract_phone(text):

    pattern = r'(\+91[\s-]?)?[6-9]\d{9}'

    match = re.search(pattern, text)

    if match:
        return match.group()

    return "Not Found"

In [108]:

print(extract_name(resume_text))
print(extract_email(resume_text))
print(extract_phone(resume_text))

Bobbili Sri Harsha
harshabob2006@gmail.com
+91-8977620064


In [103]:
from google.colab import files

uploaded = files.upload()

Saving Sri_Harsha_Resume_Final.pdf to Sri_Harsha_Resume_Final.pdf


In [104]:
filename = list(uploaded.keys())[0]

print(filename)

Sri_Harsha_Resume_Final.pdf


In [105]:
resume_text = extract_text_from_pdf(filename)

print(resume_text[:500])

BOBBILI SRI HARSHA
+91-8977620064 | harshabob2006@gmail.com | LinkedIn: linkedin.com/in/sri-harsha-bobbili-30b2152a4/
EDUCATION
Sreenidhi University, Hyderabad, India 2024 – 2028
B.Tech in Computer Science & Engineering
CGPA: 8.29 / 10
Narayana Junior College, Hyderabad 2022 – 2024
Higher Secondary Education
Score: 96%
Johnson Grammar School, Hyderabad 2022
Secondary Education
Score: 88%
PROJECTS
SmartNotes | Academic Project | Python, Flask | 2026
• Developed a note-taking platform with authent


In [106]:
career = """
Looking for an AI Engineer role
"""

job_description = """
Python
Machine Learning
TensorFlow
PyTorch
FastAPI
SQL
AWS
Docker

Minimum 2 years experience

Develop AI applications
Deploy ML models
"""

In [107]:
predict_match(
    resume_text,
    career,
    job_description,
    resume_text
)

              AI RESUME MATCH REPORT

🎯 Match Score : 74.20%

-------------------------------
🧠 Semantic Similarity : 0.288
🎯 Career Similarity   : 0.432
📝 BLEU Score          : 0.001
💼 Experience          : 0 years

🟡 Recommendation : Good Match

📌 Missing Skills
- aws
- docker
- fastapi
- machine learning
- pytorch
- sql
- tensorflow


0.7420307397842407